# exp-06 — the full figure checklist, fanned in from one entry point

Reads the runs under `experiments/runs/exp-06-full-checklist/`, scores every
leaf not scored yet, and describes the eleven-check fan-in — one entry skill,
`do-fig-checklist-cm`, one session per figure — in two arrangements. A
baseline: nothing here is a test.

**This notebook spends nothing.** The sessions are produced by

```bash
python experiments/exp-06-full-checklist/run.py
```

What is measured, and the error analysis that follows, are fixed in
[`thinking/experiments/exp-06-full-checklist.md`](../../thinking/experiments/exp-06-full-checklist.md).

## The two arrangements

| arrangement | arm directory | each check |
|---|---|---|
| `A\|B <- C_i <- D` | `pinned` | invokes `classify-panels` v1 |
| `A <- B <- C_i <- D` | `classify-panels@v2` | invokes `classify-panels` v2, which invokes `identify-panels` |

One contract, check-major: `{<check>: [rows]}` for the eleven checks, rewritten
below into per-check terms — `check`, `outputs[].x`, `outputs`.

## The gold

**`gold-v5`**. This cell pins the scorer to it and must run first.

In [ ]:
from soda_mmqc.gold import pin_gold

GOLD_TAG = "gold-v5"
pin_gold(GOLD_TAG)

In [ ]:
import importlib.util
import json
import os
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

from soda_mmqc.config import gold_root
from soda_mmqc.core.run_layout import iter_leaves
from soda_mmqc.core.scoring import ANALYSIS_FILENAME, score_check
from soda_mmqc.reporting import (
    arm_contrast,
    arm_levels,
    layer1_counts_by_example,
    layer2_counts,
    layer_s_counts_by_example,
    load_run_root,
    non_response_counts,
    plot_arm_levels_by_check,
    plot_stacked_counts,
    rate_contrast,
    scores_frame,
)
from soda_mmqc.reporting.styles import (
    LAYER1_ORDER,
    LAYER2_BINARY_ORDER,
    LAYER2_GRADED_ORDER,
    LAYER_S_ORDER,
)
from soda_mmqc.reporting.tables import LAYER1_OUTLIER_OUTCOMES, LAYER2_ERROR_OUTCOMES

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 80)


def _load(name: str, path: Path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


exp06 = _load("exp06_run", Path("../../experiments/exp-06-full-checklist/run.py"))
panels = _load("exp04_panels", Path("../../experiments/exp-04-panel-major/panels.py"))
assert exp06.GOLD_TAG == GOLD_TAG, "the runner and this notebook name different gold"

CHECKLIST, ENTRY, CHECKS = exp06.CHECKLIST, exp06.ENTRY, exp06.CHECKS
CLASSIFY, IDENTIFY = exp06.CLASSIFY, exp06.IDENTIFY
ARRANGEMENTS = tuple(exp06.ARRANGEMENTS)
LABEL_TO_ARRANGEMENT = {exp06.arm_label(u): name for name, u in exp06.ARRANGEMENTS.items()}
AB, A_B = ARRANGEMENTS

#: The full run; set EXP06_ROOT to `.../smoke` to exercise the notebook.
ROOT = Path(os.environ.get("EXP06_ROOT", "../../experiments/runs/exp-06-full-checklist"))
GOLD = gold_root()
SEED = 20261011
RESAMPLES = 10_000


def owning_check(value: str) -> str | None:
    for check in CHECKS:
        if value == check or value.startswith((f"{check}[", f"{check}.")):
            return check
    return None


def to_per_check(frame: pd.DataFrame) -> pd.DataFrame:
    """`<check>[].x` -> (`check`, `outputs[].x`); list key `<check>` -> `outputs`; arm -> arrangement."""
    frame = frame.copy()
    frame["condition"] = [LABEL_TO_ARRANGEMENT[a] for a in frame["arm"]]
    key = "property" if "property" in frame else "list_key"
    frame["check"] = [owning_check(v) for v in frame[key]]
    nested = frame["check"].isna()
    if key == "list_key" and nested.any():
        # plot-axis-units' per-axis lists keep their own keys.
        frame.loc[nested, "check"] = "plot-axis-units"
    if frame["check"].isna().any():
        raise ValueError(f"{key}s owned by no check: "
                         f"{sorted(frame.loc[frame['check'].isna(), key].unique())}")
    for column in ("property", "list_key"):
        if column in frame:
            frame[column] = ["outputs" + v[len(c):] if v.startswith(c) else v
                             for v, c in zip(frame[column], frame["check"])]
    return frame


def bootstrap_mean(values, rng) -> tuple[float, float, float, int]:
    v = np.asarray([x for x in values if not pd.isna(x)], dtype=float)
    if not len(v):
        return (np.nan, np.nan, np.nan, 0)
    boots = rng.choice(v, size=(RESAMPLES, len(v)), replace=True).mean(axis=1)
    return (v.mean(), np.percentile(boots, 2.5), np.percentile(boots, 97.5), len(v))


print(ROOT.resolve())

## Score every leaf that is not scored yet

In [ ]:
stale = []
for arm, replicate, leaf in iter_leaves(ROOT):
    analysis = leaf / ANALYSIS_FILENAME
    if not analysis.is_file():
        score_check(CHECKLIST, ENTRY, leaf, save=True)
        print(f"scored {LABEL_TO_ARRANGEMENT[arm]} rep-{replicate:02d}")
        continue
    predictions = list(leaf.rglob("prediction.json"))
    gold = [GOLD / p.parent.relative_to(leaf) / "checks" / ENTRY / "expected_output.json"
            for p in predictions]
    newest = max((p.stat().st_mtime for p in predictions + gold if p.is_file()), default=0.0)
    if newest > analysis.stat().st_mtime:
        stale.append(leaf)
if stale:
    raise RuntimeError("predictions or gold newer than their analysis in:\n  "
                       + "\n  ".join(map(str, stale)))
print("all leaves scored, none stale")

## 0. Dispatch, read first

Per session, from `skill_trace.json`: how many of the eleven checks were
invoked and which were not; how often the shared skills were invoked; whether
the checks ran in D's order; any skill outside the closure.

In [ ]:
def read_trace(path: Path) -> list[str]:
    try:
        entries = json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return []
    return [e.get("skill") for e in entries if isinstance(e, dict) and e.get("skill")]


rows = []
for arm, replicate, leaf in iter_leaves(ROOT):
    arrangement = LABEL_TO_ARRANGEMENT[arm]
    expected = {ENTRY, *CHECKS, *exp06.EXPECTED_SHARED[arrangement]}
    for trace in leaf.rglob("intermediates/skill_trace.json"):
        skills = read_trace(trace)
        called = list(dict.fromkeys(s for s in skills if s in CHECKS))
        rows.append({
            "condition": arrangement, "replicate": replicate,
            "example": str(trace.parent.parent.relative_to(leaf)),
            "n_checks": len(called),
            **{c: c in called for c in CHECKS},
            "classify_calls": skills.count(CLASSIFY),
            "identify_calls": skills.count(IDENTIFY),
            "in_order": called == [c for c in CHECKS if c in called],
            "spontaneous": tuple(sorted(set(skills) - expected)),
        })
sessions = pd.DataFrame(rows)
print("checks invoked per session:")
print(sessions.groupby(["condition", "n_checks"]).size().unstack(fill_value=0))

In [ ]:
# Per check, in D's order: the share of sessions that invoked it. A fall-off
# along the list would say position matters.
sessions.groupby("condition")[list(CHECKS)].mean().T

In [ ]:
for col in ("classify_calls", "identify_calls"):
    print(col)
    print(sessions.groupby(["condition", col]).size().unstack(fill_value=0))
    print()
print("in D's order:", sessions.groupby("condition")["in_order"].agg(["sum", "count"]).to_dict("index"))
spont = sessions.explode("spontaneous").dropna(subset=["spontaneous"])
print("spontaneous skills:", "none" if spont.empty else spont.groupby(["condition", "spontaneous"]).size().to_dict())

## One row per property, per replicate, per example

In [ ]:
runs = load_run_root(ROOT, checklist=CHECKLIST, check=ENTRY)
scores = to_per_check(scores_frame(runs))
layer_s = to_per_check(layer_s_counts_by_example(runs))
layer_1 = to_per_check(layer1_counts_by_example(runs))
layer_2 = to_per_check(layer2_counts(runs))
non_response = to_per_check(non_response_counts(runs))
print(f"{len(scores)} score rows; replicates per condition:",
      scores.groupby("condition")["replicate"].nunique().to_dict())

## Failures and empty answers

A failed session loses all eleven checks.

In [ ]:
print(non_response[non_response["list_key"] == "outputs"]
      .groupby(["check", "condition"])["empty"].sum().unstack(fill_value=0))
wanted = json.loads((Path("../../soda_mmqc/data/checklist") / CHECKLIST / ENTRY / "benchmark.json")
                    .read_text(encoding="utf-8"))["examples"]
missing = [{"condition": LABEL_TO_ARRANGEMENT[arm], "replicate": rep,
            "without_prediction": sum(not (leaf / ex / "prediction.json").is_file() for ex in wanted)}
           for arm, rep, leaf in iter_leaves(ROOT)]
pd.DataFrame(missing).groupby("condition")["without_prediction"].sum()

## The outcomes, arrangement beside arrangement

In [ ]:
plot_stacked_counts(
    layer_s[layer_s['list_key'] == 'outputs'], group="check", category="outcome", series="condition",
    order=LAYER_S_ORDER, series_order=ARRANGEMENTS, ylabel="rows",
    title="Layer S row outcomes, summed over examples and replicates",
)

In [ ]:
plot_stacked_counts(
    layer_1, group="check", category="layer1", series="condition",
    order=LAYER1_ORDER, series_order=ARRANGEMENTS, ylabel="instances",
    title="Layer 1 applicability calls, summed over examples and replicates",
)

In [ ]:
plot_stacked_counts(
    layer_2[layer_2['layer2'].isin(LAYER2_BINARY_ORDER)], group="check", category="layer2", series="condition",
    order=LAYER2_BINARY_ORDER, series_order=ARRANGEMENTS, ylabel="instances",
    title="Layer 2, binary properties, summed over examples and replicates",
)

In [ ]:
plot_stacked_counts(
    layer_2[layer_2['layer2'].isin(LAYER2_GRADED_ORDER)], group="check", category="layer2", series="condition",
    order=LAYER2_GRADED_ORDER, series_order=ARRANGEMENTS, ylabel="instances",
    title="Layer 2, graded and multiclass properties, summed over examples and replicates",
)

## Layer S, and panels across the eleven lists

Per check, `correct / (correct + missing)` and
`correct / (correct + missing + spurious)`: mean of per-figure rates,
replicates pooled first, bootstrap over figures. Then, per figure, the panels
stated across all eleven check lists against the gold's.

In [ ]:
def per_example_rate(frame, category, numerator, denominator=None):
    counts = (frame.groupby(["condition", "check", "example", category])["count"].sum()
              .unstack(fill_value=0))
    num = counts[[c for c in numerator if c in counts]].sum(axis=1)
    den = counts[[c for c in (denominator or counts.columns) if c in counts]].sum(axis=1)
    return (num / den.where(den > 0)).rename("rate").reset_index()


rng = np.random.default_rng(SEED)
rows = []
rows_s = layer_s[layer_s["list_key"] == "outputs"]
for label, den in (("found", ("correct_row", "missing_row")),
                   ("both errors", ("correct_row", "missing_row", "spurious_row"))):
    r = per_example_rate(rows_s, "outcome", ("correct_row",), den)
    for (cond, check), sub in r.groupby(["condition", "check"]):
        mean, low, high, n = bootstrap_mean(sub["rate"], rng)
        rows.append({"rate": label, "condition": cond, "check": check, "mean": mean,
                     "ci_low": low, "ci_high": high, "n_examples": n})
layer_s_rates = pd.DataFrame(rows)
layer_s_rates.pivot_table(index="check", columns=["rate", "condition"], values="mean").reindex(CHECKS)

In [ ]:
rows = []
for arm, replicate, leaf in iter_leaves(ROOT):
    for prediction in leaf.rglob("prediction.json"):
        example = str(prediction.parent.relative_to(leaf))
        gold = json.loads((GOLD / example / "checks" / ENTRY / "expected_output.json")
                          .read_text(encoding="utf-8"))
        labels = [r["panel_label"] for r in gold[CHECKS[0]]]
        rows.append({"condition": LABEL_TO_ARRANGEMENT[arm], "replicate": replicate,
                     "example": example,
                     **panels.panel_counts(labels, json.loads(prediction.read_text()), "cm", CHECKS)})
panel_rows = pd.DataFrame(rows)
totals = panel_rows.groupby("condition")[["spurious_panels", "missing_panels",
                                         "partly_missing_panels", "gold_panels"]].sum()
totals["sessions"] = panel_rows.groupby("condition").size()
totals["figures_with_spurious"] = (panel_rows[panel_rows["spurious_panels"] > 0]
                                   .groupby("condition")["example"].nunique())
totals.fillna(0)

## Layer 1, per check

In [ ]:
rng = np.random.default_rng(SEED)
r = per_example_rate(layer_1, "layer1", ("correct_applicable", "correct_NA"))
rows = []
for (cond, check), sub in r.groupby(["condition", "check"]):
    mean, low, high, n = bootstrap_mean(sub["rate"], rng)
    rows.append({"condition": cond, "check": check, "mean": mean, "ci_low": low,
                 "ci_high": high, "n_examples": n})
layer_1_rates = pd.DataFrame(rows)
layer_1_rates.pivot_table(index="check", columns="condition", values=["mean", "ci_low", "ci_high"]).reindex(CHECKS)

## Layer 2, per check and per property

`mean_score` on applicable instances: per figure, averaged over replicates;
then the mean over figures with a bootstrap interval.

In [ ]:
rng = np.random.default_rng(SEED)
per_fig = (scores.dropna(subset=["mean_score"])
           .groupby(["condition", "check", "property", "example"])["mean_score"].mean().reset_index())
rows = []
for (cond, check, prop), sub in per_fig.groupby(["condition", "check", "property"]):
    mean, low, high, n = bootstrap_mean(sub["mean_score"], rng)
    rows.append({"condition": cond, "check": check, "property": prop, "mean": mean,
                 "ci_low": low, "ci_high": high, "n_examples": n})
layer_2_props = pd.DataFrame(rows)
layer_2_props["check"] = pd.Categorical(layer_2_props["check"], CHECKS, ordered=True)
layer_2_props.pivot_table(index=["check", "property"], columns="condition",
                          values=["mean", "n_examples"], observed=True)

In [ ]:
# One panel per check, one bar per property per arrangement, over the figures
# both arrangements answered.
levels = []
for check in CHECKS:
    sub = scores[scores["check"] == check].drop(columns=["arm"]).rename(columns={"condition": "arm"})
    if {AB, A_B} <= set(sub["arm"]):
        levels.append(arm_levels(sub, baseline=AB, variant=A_B))
levels = pd.concat(levels, ignore_index=True) if levels else pd.DataFrame()
plot_arm_levels_by_check(levels, series="arm", columns=2,
                         title="Layer 2 mean_score per property, per check")

## Layer 2, each check's decision

One decision-like property per check -- its `decision`, or what plays that
part -- read on its own rather than averaged with the check's other
properties. `mean_score` on applicable instances, per figure averaged over
replicates, then the mean over figures with a bootstrap interval (the table
above, restricted).

In [ ]:
#: One decision-like property per check: its decision, or what plays that part.
DECISIONS = {
    "micrograph-scale-bar": "scale_bar_defined_in_caption",
    "image-annotation-defined": "annotation_defined_in_caption",
    "single-channel-for-overlay": "all_channels_shown_separately",
    "individual-data-points": "decision",
    "error-bars-defined": "decision",
    "plot-axis-units": "decision",
    "plot-gap-labeling": "decision",
    "stat-significance-level": "decision",
    "stat-test": "decision",
    "replication-reporting": "decision",
    "panel-image-matches-caption": "image_matches_caption",
}
assert set(DECISIONS) == set(CHECKS)


def is_decision(check: str, prop: str) -> bool:
    return DECISIONS.get(check) == prop.removeprefix("outputs[].")


decisions = layer_2_props[[is_decision(c, p) for c, p in
                           zip(layer_2_props["check"], layer_2_props["property"])]].copy()
decisions["check"] = decisions["check"].astype(str)
decisions.pivot_table(index="check", columns="condition",
                      values=["mean", "ci_low", "ci_high", "n_examples"]).reindex(CHECKS).round(3)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 0.5 * len(CHECKS) + 1.5))
height = 0.38
for i, cond in enumerate(ARRANGEMENTS):
    sub = decisions[decisions["condition"] == cond].set_index("check").reindex(CHECKS)
    y = np.arange(len(CHECKS)) + (i - 0.5) * height
    ax.barh(y, sub["mean"], height=height, label=cond,
            xerr=[sub["mean"] - sub["ci_low"], sub["ci_high"] - sub["mean"]], capsize=2)
ax.set_yticks(np.arange(len(CHECKS)))
ax.set_yticklabels([f"{c} · {DECISIONS[c]}" for c in CHECKS])
ax.invert_yaxis()
ax.set_xlim(min(0.7, decisions["ci_low"].min() - 0.02), 1.005)
ax.set_xlabel("mean_score, mean over figures (95% bootstrap interval)")
ax.set_title("Layer 2, each check's decision")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=False)
fig.tight_layout()

## Cost and time per figure

At concurrency 50; comparable only with runs at the same concurrency.

In [ ]:
def read_usage(path: Path) -> dict:
    audit = json.loads(path.read_text(encoding="utf-8"))
    usage = audit.get("usage") or {}
    tokens = usage.get("usage") or {}
    return {
        "concurrency": (audit.get("session") or {}).get("concurrency"),
        "cost": usage.get("total_cost_usd"),
        "turns": usage.get("num_turns"),
        "seconds": (usage.get("duration_ms") or 0) / 1000,
        "input_tokens": sum(tokens.get(k) or 0 for k in
                            ("input_tokens", "cache_creation_input_tokens", "cache_read_input_tokens")),
        "cache_read_tokens": tokens.get("cache_read_input_tokens") or 0,
        "output_tokens": tokens.get("output_tokens") or 0,
    }


usage = pd.DataFrame([{"condition": LABEL_TO_ARRANGEMENT[arm], "replicate": rep,
                       "example": str(a.parent.parent.relative_to(leaf)), **read_usage(a)}
                      for arm, rep, leaf in iter_leaves(ROOT)
                      for a in leaf.rglob("intermediates/tool_audit.json")])
print("concurrency recorded:", usage["concurrency"].value_counts().to_dict())
rng = np.random.default_rng(SEED)
per_figure = usage.groupby(["condition", "example"]).mean(numeric_only=True).reset_index()
rows = []
for cond, sub in per_figure.groupby("condition"):
    for q in ("cost", "input_tokens", "cache_read_tokens", "output_tokens", "turns", "seconds"):
        mean, low, high, n = bootstrap_mean(sub[q], rng)
        rows.append({"condition": cond, "quantity": q, "mean": mean, "ci_low": low,
                     "ci_high": high, "n_figures": n})
print(f"total cost ${usage['cost'].sum():.2f} over {len(usage)} sessions")
pd.DataFrame(rows).pivot_table(index="quantity", columns="condition", values=["mean", "ci_low", "ci_high"])

## Sessions that invoked no check

Four sessions -- one in `A|B <- C_i <- D`, three in `A <- B <- C_i <- D` --
answered all eleven checks without invoking any. Each is set beside the same
figure's other replicates in the same arrangement, which all dispatched: layer
S and layer 1 pooled over the eleven checks; at layer 2, the errors on the
eleven decisions above, counted rather than averaged across properties; and
the session's cost. Four sessions: read them as cases, not as an estimate.

In [ ]:
KEY = ["condition", "replicate", "example"]


def session_rates() -> pd.DataFrame:
    # Layer S and layer 1 pooled over the eleven checks. Layer 2 is not
    # averaged across properties: it counts errors on the eleven decisions
    # (1 - score per instance; a list-valued decision contributes fractions).
    s = (layer_s[layer_s["list_key"] == "outputs"].groupby(KEY + ["outcome"])["count"].sum()
         .unstack(fill_value=0))
    out = pd.DataFrame({
        "S_found": s["correct_row"] / (s["correct_row"] + s["missing_row"]),
        "S_both": s["correct_row"] / s[["correct_row", "missing_row", "spurious_row"]].sum(axis=1),
        "spurious_rows": s["spurious_row"], "missing_rows": s["missing_row"],
    })
    l1 = layer_1.groupby(KEY + ["layer1"])["count"].sum().unstack(fill_value=0)
    out["layer_1"] = l1[["correct_applicable", "correct_NA"]].sum(axis=1) / l1.sum(axis=1)
    d = scores.dropna(subset=["mean_score"])
    d = d[[is_decision(c, p) for c, p in zip(d["check"], d["property"])]]
    d = d.assign(errors=(1 - d["mean_score"]) * d["n_scored"])
    out["decision_errors"] = d.groupby(KEY)["errors"].sum()
    out["decision_instances"] = d.groupby(KEY)["n_scored"].sum()
    out = out.join(usage.set_index(KEY)[["output_tokens", "turns", "cost"]])
    return out.join(sessions.set_index(KEY)[["n_checks", "classify_calls"]]).reset_index()


per_session = session_rates()
zero = per_session[per_session["n_checks"] == 0]
# Each zero-check session beside the same figure's other replicates in the
# same arrangement -- the sessions that dispatched.
columns = ["S_found", "S_both", "spurious_rows", "missing_rows", "layer_1", "decision_errors",
           "decision_instances", "output_tokens", "turns", "cost", "classify_calls"]
rows = []
for _, z in zero.iterrows():
    same = per_session[(per_session["condition"] == z["condition"])
                       & (per_session["example"] == z["example"])
                       & (per_session["replicate"] != z["replicate"])]
    rows.append({"condition": z["condition"], "example": z["example"], "session": "no check",
                 "replicate": z["replicate"], **z[columns].to_dict()})
    rows.append({"condition": z["condition"], "example": z["example"],
                 "session": f"dispatched (mean of {len(same)})", "replicate": None,
                 **same[columns].mean().to_dict()})
zero_vs_dispatched = pd.DataFrame(rows).set_index(["condition", "example", "session"])
zero_vs_dispatched.round(3)

In [ ]:
# Per check, the four zero-check sessions pooled against their figures'
# dispatched replicates: layer 1, and layer 2 on the check's decision.
def per_check(frame_1, frame_2) -> pd.DataFrame:
    l1 = frame_1.groupby(["check", "layer1"])["count"].sum().unstack(fill_value=0)
    d = frame_2.dropna(subset=["mean_score"])
    d = d[[is_decision(c, p) for c, p in zip(d["check"], d["property"])]]
    return pd.DataFrame({"layer_1": l1[["correct_applicable", "correct_NA"]].sum(axis=1) / l1.sum(axis=1),
                         "decision": d.groupby("check")["mean_score"].mean()})


zero_keys = set(map(tuple, zero[KEY].itertuples(index=False)))
figures = set(map(tuple, zero[["condition", "example"]].itertuples(index=False)))


def pick(frame, zero_sessions: bool):
    keys = list(zip(frame["condition"], frame["replicate"], frame["example"]))
    in_zero = np.array([k in zero_keys for k in keys])
    same_figure = np.array([(c, e) in figures for c, _, e in keys])
    return frame[in_zero if zero_sessions else (same_figure & ~in_zero)]


by_check = per_check(pick(layer_1, True), pick(scores, True)).join(
    per_check(pick(layer_1, False), pick(scores, False)), lsuffix=" no check", rsuffix=" dispatched")
by_check.reindex(CHECKS).round(3)

## Replicate variance, for later margins

The half-width a later five-against-five comparison would plan for:
`1.96 · sqrt(mean_e(2σ²_e / 5) / n_e)`.

In [ ]:
def replicate_variance(frame, category, numerator, denominator=None):
    counts = (frame.groupby(["condition", "check", "example", "replicate", category])["count"].sum()
              .unstack(fill_value=0))
    num = counts[[c for c in numerator if c in counts]].sum(axis=1)
    den = counts[[c for c in (denominator or counts.columns) if c in counts]].sum(axis=1)
    rate = (num / den.where(den > 0)).rename("rate").reset_index()
    var = rate.groupby(["condition", "check", "example"])["rate"].var(ddof=1).rename("var").reset_index()
    out = var.groupby(["condition", "check"]).agg(mean_var=("var", "mean"), n_examples=("var", "count"))
    out["planned_half_width_5v5"] = 1.96 * np.sqrt(2 * out["mean_var"] / 5 / out["n_examples"])
    return out


pd.concat({
    "layer S": replicate_variance(rows_s, "outcome", ("correct_row",), ("correct_row", "missing_row")),
    "layer 1": replicate_variance(layer_1, "layer1", ("correct_applicable", "correct_NA")),
}, names=["endpoint"])

## The arrangements side by side

`A <- B <- C_i <- D` minus `A|B <- C_i <- D`, paired by figure. **Reported, not
tested**: there is no margin and no verdict.

In [ ]:
rows = []
for label, frame, category, numerator, denominator, group in (
    ("layer S", rows_s, "outcome", ("correct_row",), ("correct_row", "missing_row"), ("check", "list_key")),
    ("layer 1", layer_1, "layer1", ("correct_applicable", "correct_NA"), None, ("check",)),
):
    for check in CHECKS:
        pair = frame[frame["check"] == check]
        if pair["condition"].nunique() < 2:
            continue
        c = rate_contrast(pair.drop(columns=["arm"]).rename(columns={"condition": "arm"}),
                          baseline=AB, variant=A_B, category=category, numerator=numerator,
                          denominator=denominator, group=group, resamples=RESAMPLES, seed=SEED)
        c.insert(0, "endpoint", label)
        rows.append(c)
contrast_s1 = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
contrast_s1[["endpoint", "check", "difference", "ci_low", "ci_high", "n_examples"]] if rows else contrast_s1

In [ ]:
rows = []
for check in CHECKS:
    sub = scores[scores["check"] == check].drop(columns=["arm"]).rename(columns={"condition": "arm"})
    if {AB, A_B} <= set(sub["arm"]):
        rows.append(arm_contrast(sub, baseline=AB, variant=A_B))
contrast_2 = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
if not contrast_2.empty:
    contrast_2["ci_low"] = contrast_2["difference"].astype(float) - 1.96 * contrast_2["se"].astype(float)
    contrast_2["ci_high"] = contrast_2["difference"].astype(float) + 1.96 * contrast_2["se"].astype(float)
contrast_2.sort_values("difference")[["check", "property", "difference", "ci_low", "ci_high",
                                      "n_examples", "paired_fraction"]] if not contrast_2.empty else contrast_2

## Error analysis: the candidates

Every layer-1 applicability error and layer-2 disagreement, from the scored
instances, with its figure, check, panel and property, and the gold and
predicted values. Then:

1. **per check and property**, how many there are;
2. **recurring disagreements**: the same (figure, check, panel, property)
   wrong in several sessions, with the most common predicted value — the ones
   to read first.

The recurring ones are written to
`experiments/exp-06-full-checklist/error_analysis_candidates.csv`, with empty
`sorting` and `note` columns for the manual step: **model error**, **gold
error** or **ambiguous**. The file is rewritten on every run of this cell, so
annotate a copy, not the file itself.

In [ ]:
def row_index(path: str) -> int | None:
    head = path.split("].", 1)[0]
    return int(head.rsplit("[", 1)[1]) if "[" in head else None


rows = []
for arm, replicate, leaf in iter_leaves(ROOT):
    analysis = json.loads((leaf / ANALYSIS_FILENAME).read_text(encoding="utf-8"))
    for record in analysis.get("flat", []):
        example = (record.get("metadata") or {}).get("source")
        gold = record.get("expected_output") or {}
        for inst in (record.get("analysis") or {}).get("instances", []):
            l1, l2 = inst.get("layer1"), inst.get("layer2")
            if l1 not in LAYER1_OUTLIER_OUTCOMES and l2 not in LAYER2_ERROR_OUTCOMES:
                continue
            check = owning_check(inst["leaf_property"])
            i = row_index(inst["path"])
            gold_rows = gold.get(check) or []
            panel = gold_rows[i].get("panel_label") if i is not None and i < len(gold_rows) else None
            rows.append({
                "condition": LABEL_TO_ARRANGEMENT[arm], "replicate": replicate,
                "example": example, "check": check, "panel": panel,
                "property": "outputs" + inst["leaf_property"][len(check):],
                "layer": "layer 1" if l1 in LAYER1_OUTLIER_OUTCOMES else "layer 2",
                "outcome": l1 if l1 in LAYER1_OUTLIER_OUTCOMES else l2,
                "exp_value": json.dumps(inst.get("exp_value"), ensure_ascii=False),
                "pred_value": json.dumps(inst.get("pred_value"), ensure_ascii=False),
            })
errors = pd.DataFrame(rows)
print(f"{len(errors)} error instances")
(errors.groupby(["check", "property", "layer", "condition"]).size().unstack(fill_value=0)
 .assign(total=lambda f: f.sum(axis=1)).sort_values("total", ascending=False))

In [ ]:
sessions_per_condition = sessions.groupby("condition").size() / sessions["example"].nunique()
recurring = (errors.groupby(["condition", "example", "check", "panel", "property", "layer", "exp_value"])
             .agg(sessions_wrong=("replicate", "nunique"),
                  most_common_pred=("pred_value", lambda s: Counter(s).most_common(1)[0][0]),
                  outcome=("outcome", lambda s: Counter(s).most_common(1)[0][0]))
             .reset_index()
             .sort_values(["sessions_wrong", "check"], ascending=[False, True]))
recurring["sorting"] = ""
recurring["note"] = ""
out = Path("../../experiments/exp-06-full-checklist/error_analysis_candidates.csv")
n_recurring = int((recurring["sessions_wrong"] >= 3).sum())
if "EXP06_ROOT" in os.environ:
    print(f"{n_recurring} recurring disagreements; not written -- EXP06_ROOT points away from the full run")
else:
    recurring[recurring["sessions_wrong"] >= 3].to_csv(out, index=False)
    print(f"{n_recurring} disagreements wrong in at least 3 of "
          f"{int(sessions_per_condition.max())} sessions, written to {out.name}")
recurring.head(40)

## What this does not do

It does not sort the disagreements, and it does not decide what to improve:
that is the manual error analysis, recorded in
[`thinking/experiments/exp-06-full-checklist.md`](../../thinking/experiments/exp-06-full-checklist.md).